# What changes along the myeloma precursor sequence, read off the model's own units

Two independent bone-marrow T-cell datasets span the full precursor sequence —
non-cancer, MGUS, smouldering, symptomatic. This asks what a pathway-informed
VAE says changes along it, *inside each dataset separately*, and it asks it
before consulting either source paper. The literature check happens afterwards,
in `FINDINGS.md`.

**What changed, and why this notebook replaced `09_model_biology.ipynb`.** That
notebook read the stage effect out of a dense `(n_genes, n_cov)` covariate term
added *after* the pathway mask, then scored the resulting per-gene shifts with
a competitive enrichment test over Hallmark. Two things were wrong with that,
both measured in `01_architecture.py`:

- the annotation was not in the contrast path at all, so a **shuffled**
  gene→pathway map reproduced the result at Spearman +0.97;
- the model's own units ranked the expected programme **1,611th to 1,615th of
  1,615**, because `dec_out` was half negative and a unit rising pushed half
  its genes up and half down.

Here the covariate enters *inside* the pathway layer, is the only route, and
`dec_out` is non-negative. The stage effect is then one signed number per
annotated unit — `InformedVAE.pathway_contrast` — with no cells re-encoded, no
enrichment test anywhere, and a sign that means what it says.

**Units only.** There is no Hallmark panel and no gene-level read-out in this
notebook. The gene-level test survives only inside `01_architecture.py`, as the
evidence that it was annotation-blind.

| | |
| --- | --- |
| **Reads** | `data/Tcell.h5ad`, `data/metadata-donor.csv`, `resources/c2.cp.reactome*.gmt` |
| **Writes** | `outputs/biology/<study>/` — `unit_contrasts.csv`, three tables, five figures, `summary.json` |
| **Run by** | `snakemake units`, once per study, one GPU |

## 1. Parameters and inputs

The first code cell is tagged `parameters`, so papermill replaces it when the
workflow runs this notebook once per study. Changing a default here changes the
run for both studies, which is deliberate: a parameter that differs between
them would make the two fits incomparable, and comparability is the thing this
design is built to protect.

In [ ]:
# Parameters. papermill injects overrides directly below this cell.
study = "Zavidij_2020"
epochs = 250
seed = 42

# Honour the atlas's own donor-level `analysed` flag, which the authors define as
# whether the donor entered any analysis beyond clustering and phenotyping. It
# removes 3 donors and 18,226 cells (12.7%) from Foster_2026 and nothing from
# Zavidij, where all 30 donors are flagged True.
analysed_only = True

# Reuse a saved fit when one is already in this run's output directory and was
# produced by exactly this design. Set True to force training regardless. The
# fingerprint is checked, not just the filename: a checkpoint from a different
# study, epoch count, seed, cell set, architecture or covariate design is never
# reused, so iterating on a figure costs seconds instead of half an hour and
# cannot silently read back the wrong model.
force_retrain = False

# Candidate covariate blocks, in the order they are audited. The stage block
# (cohort x lineage) is always present -- it is the thing being estimated. Every
# other name here is a *candidate*: section 2 checks it and only the ones that
# pass are built into the design matrix.
#
# The default is `pheno_h` alone, which is the smallest set that makes the stage
# contrast a within-phenotype comparison. Add "sex", "depth_bin" or "sort" to see
# them audited and included; section 2 prints why each would or would not survive
# either way. `depth_bin` is a genuine confounder in these data -- patients are
# sequenced deeper than controls -- and it passes the audit in both studies, so it
# is available here as a one-word change. It is not on by default because neither
# source paper corrects for detection depth, and a corrected result cannot be
# compared one-for-one with their uncorrected claims. There is no longer a second
# depth-adjusted fit per study: one architecture, one fit, one history.
#
# `pheno_h` rather than `pheno`: the raw atlas phenotype cannot enter Foster_2026,
# because its `Teff` level is 95.5% MM and so doubles as an MM indicator (section
# 2 catches this and prints it). `mm.PHENO_MERGE` folds the three
# lineage-agnostic clusters into one level and `CD8+Tex` into `CD8+Tte`, giving
# **16 levels that are identical in both studies** -- so the same design is
# identified in each, which is the only way the two fits are comparable.
candidate_blocks = ["pheno_h"]
depth_bins = 5          # only used if "depth_bin" is among the candidates

In [ ]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split

from pyvae import (
    InformedVAE,
    integrated_gradients,
    set_all_seeds,
    train_ivae_modern,
    unit_sign_structure,
)

import myeloma as mm

warnings.filterwarnings("ignore", category=UserWarning, module="anndata")
mm.apply_style()

STUDY = str(study)
SEED = int(seed)
set_all_seeds(SEED)
DEVICE = mm.device()

# A short run must not overwrite a real one. The workflow trains for
# CANONICAL_EPOCHS; anything less is a smoke test or an interactive experiment and
# is written to its own directory. Without this, `papermill -p epochs 3` silently
# replaces every table and figure in `outputs/biology/<study>/` with 3-epoch
# values, and the next person to read those files has no way to tell.
CANONICAL_EPOCHS = 250
SUFFIX = "" if int(epochs) >= CANONICAL_EPOCHS else f"__epochs{int(epochs)}"
CANDIDATE_BLOCKS = [str(b) for b in candidate_blocks]
OUT_DIR = Path("outputs/biology") / f"{STUDY}{SUFFIX}"
if SUFFIX:
    print(f"!! {epochs} epochs is below the canonical {CANONICAL_EPOCHS}: "
          f"writing to {OUT_DIR} and leaving the real run alone")
FIGS = OUT_DIR / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
# `01_architecture.py` writes one directory per study, because the two are
# separate fits with different cell counts and different arms.
ARCH_SUMMARY = Path("outputs/architecture") / STUDY / "summary.json"

mm.require_inputs()
adata = mm.load_tcells(STUDY, analysed_only=bool(analysed_only))
STAGES = [s for s in mm.STAGE_ORDER if s in set(adata.obs["cohort"].astype(str))]
LADDER = [(STAGES[0], s) for s in STAGES[1:]]
LADDER_NAMES = [f"{a}->{b}" for a, b in LADDER]

donors = adata.obs.groupby("cohort", observed=True)["donor_id"].nunique()
cells = adata.obs.groupby("cohort", observed=True).size()
print(f"study              {STUDY}")
print(f"device             {DEVICE}")
print(f"cells              {adata.n_obs:,}")
print(f"donors             {adata.obs['donor_id'].nunique()}")
print(f"analysed_only      {bool(analysed_only)}"
      "   (the atlas's own donor inclusion flag)")
print(f"writing to         {OUT_DIR}")
print(f"\nstage              " + "  ".join(f"{s}" for s in STAGES))
print("donors             " + "  ".join(f"{int(donors.get(s, 0))}" for s in STAGES))
print("cells              " + "  ".join(f"{int(cells.get(s, 0)):,}" for s in STAGES))
print(f"\nladder             " + ", ".join(LADDER_NAMES)
      + "   (every contrast against the healthy baseline)")

## 2. Every covariate, audited before it enters

This is the part worth reading slowly, because it is where the design either is
or is not identified. It is unchanged from the previous notebook — the
architecture moved, the design did not.

**Cohort is a property of the donor.** Every donor in both analysed studies sits
in exactly one cohort — 30 of 30 and 25 of 25, none spanning two — so the stage
effect is estimated purely from *between-donor* variation and there is no
within-donor stage replication anywhere in the design. That has a consequence
for covariates that is easy to miss:

> A covariate that does not vary **within** a donor cannot be separated from
> donor identity, and because cohort is itself a donor property, such a
> covariate competes with the stage effect for the same variance.

So the audit asks four questions of each candidate, and a block has to pass all
four: does it vary at all in this study; does it vary within donors; is it
missing anywhere in a way aligned with a cohort; and is any of its levels
concentrated in a single cohort. Candidates that were not requested are audited
anyway, so the decision is visible rather than implied.

In [ ]:
def audit_block(adata, block, *, concentration_limit=0.95):
    """Four checks on one candidate covariate. Returns (verdict, reason, row)."""
    if block not in adata.obs.columns:
        return False, "not in obs", {}
    col = adata.obs[block]
    as_str = col.astype(str)
    present = col.notna()
    levels = as_str[present].nunique()

    within = (adata.obs.loc[present].groupby("donor_id", observed=True)[block]
              .nunique(dropna=True))
    donors_varying = int((within > 1).sum())

    n_missing = int((~present).sum())
    missing_cohorts = (adata.obs.loc[~present, "cohort"].astype(str)
                       .value_counts(normalize=True))
    missing_aligned = (float(missing_cohorts.iloc[0]) if n_missing else 0.0)
    missing_where = (missing_cohorts.index[0] if n_missing else "")

    share = (pd.crosstab(as_str[present], adata.obs.loc[present, "cohort"].astype(str),
                         normalize="index").max(axis=1))
    worst_level = share.idxmax() if len(share) else ""
    worst_share = float(share.max()) if len(share) else 0.0

    row = {"block": block, "levels": levels, "donors_varying": donors_varying,
           "missing_cells": n_missing, "missing_aligned": round(missing_aligned, 3),
           "worst_level": worst_level, "worst_cohort_share": round(worst_share, 3)}

    if levels < 2:
        return False, f"only {levels} level in this study: a constant column", row
    if n_missing and missing_aligned >= concentration_limit:
        return False, (f"{n_missing:,} cells have no value and {missing_aligned:.0%} "
                       f"of them are cohort '{missing_where}' -- get_dummies would "
                       f"make this block a 'not {missing_where}' indicator and it "
                       "would absorb part of the stage effect"), row
    if worst_share >= concentration_limit:
        return False, (f"level '{worst_level}' is {worst_share:.0%} one cohort, so "
                       "it doubles as a cohort indicator"), row
    if donors_varying == 0:
        return False, ("varies between donors but never within one, so it cannot be "
                       "separated from donor identity, and cohort is a donor "
                       "property"), row
    return True, f"{levels} levels, varies within {donors_varying} donors", row


EXTRA = ["pheno", "sex", "sort", "depth_bin", "chem", "age"]
if "depth_bin" in CANDIDATE_BLOCKS + EXTRA and "depth_bin" not in adata.obs.columns:
    adata.obs["depth_bin"] = pd.qcut(
        adata.obs["n_genes"].rank(method="first"), int(depth_bins),
        labels=[f"q{k + 1}" for k in range(int(depth_bins))])

rows, accepted = [], []
print("candidates requested for this run:\n")
for block in CANDIDATE_BLOCKS:
    ok, reason, row = audit_block(adata, block)
    rows.append({**row, "requested": True, "verdict": "ENTERS" if ok else "DROPPED"})
    print(f"  {'ENTERS ' if ok else 'DROPPED'} {block:11s} {reason}")
    if ok:
        accepted.append(block)

print("\nnot requested, audited anyway so the decision is visible:\n")
for block in EXTRA:
    if block in CANDIDATE_BLOCKS:
        continue
    ok, reason, row = audit_block(adata, block)
    if not row:
        continue
    rows.append({**row, "requested": False,
                 "verdict": "would enter" if ok else "would be DROPPED"})
    print(f"  {'would enter    ' if ok else 'would be DROPPED'} {block:11s} {reason}")

audit = pd.DataFrame(rows).set_index("block")
audit.to_csv(OUT_DIR / "covariate_audit.csv")
print(f"\naccepted blocks    {accepted}")
print(f"stage block        {mm.STAGE_BLOCK} (cohort x lineage), always present")

### Building the design matrix from exactly what passed

The stage block is the cohort-by-lineage interaction, which is what makes a
stage contrast *inside* one lineage a single weight difference — now a
difference of two columns of `cov_path`, in pathway coordinates rather than
gene coordinates.

In [ ]:
adata.obs[mm.STAGE_BLOCK] = (adata.obs["subset"].astype(str) + "__"
                             + adata.obs["cohort"].astype(str))
blocks = [mm.STAGE_BLOCK] + accepted
cov = pd.get_dummies(adata.obs[blocks].astype(str), dtype=np.float32)
assert cov.sum(axis=1).min() == len(blocks), (
    "a cell has fewer one-hot entries than there are blocks, so some block still "
    "has a missing value -- the audit should have dropped it")
n_cov = cov.shape[1]

print(f"design matrix      {cov.shape[0]:,} cells x {n_cov} columns")
for block in blocks:
    cols = [c for c in cov.columns if c.startswith(f"{block}_")]
    print(f"  {block + '_':<8s} {len(cols):>3d} columns")

adata, adj_df, gene_names, pathway_names = mm.build_pathways(adata)
cov = cov.loc[adata.obs_names]
cov_columns = list(cov.columns)
x_trans = adata.to_df().astype(np.float32)
counts_df = adata.to_df(layer="counts").astype(np.float32)
print(f"\npathway map        {len(gene_names):,} genes x {len(pathway_names):,} "
      f"units")

strat = adata.obs["cohort"].astype(str) + "_" + adata.obs["pheno_h"].astype(str)
if strat.value_counts().min() < 2:
    strat = adata.obs["cohort"].astype(str)
BATCH_SIZE = mm.BATCH_SIZE
idx_tr, idx_va = train_test_split(np.arange(len(adata)), test_size=0.2,
                                  stratify=strat, random_state=SEED)
# A training split that is 1 mod the batch size leaves a trailing batch of one
# row, and BatchNorm raises on it. It is not hypothetical: honouring `analysed`
# puts Foster_2026 at 100,481 training cells, which is 128 x 785 + 1.
if len(idx_tr) % BATCH_SIZE == 1:
    idx_va = np.append(idx_va, idx_tr[-1])
    idx_tr = idx_tr[:-1]
    print("moved 1 cell to validation so the last training batch is not size 1")
data = {"x_train": x_trans.iloc[idx_tr], "x_val": x_trans.iloc[idx_va],
        "counts_train": counts_df.iloc[idx_tr], "counts_val": counts_df.iloc[idx_va],
        "cov_train": cov.iloc[idx_tr], "cov_val": cov.iloc[idx_va]}
print(f"train / val        {len(idx_tr):,} / {len(idx_va):,} cells")

## 3. Train, under the architecture `01_architecture.py` selected

The architecture lives in `myeloma.py` so this notebook, `01_architecture.py`
and `03_donor_test.py` cannot drift apart: a donor-level test of this read-out
is only a test of *this* read-out if it fits the same model. The decoder is

```
logits = dec_out(tanh(dec_latent(z) + cov_path(cov)))
```

with `dec_out` masked by Reactome and reparameterised as `softplus(theta)`, and
with no covariate route into the encoder. Three consequences, all of which
section 4 checks rather than assumes:

- the covariate reaches a gene **only** through the annotation;
- `cov_path` is the **whole** conditional effect, not one of two additive
  routes, so there is no counterfactual convention left to choose;
- a unit rising raises every one of its member genes, so the sign of a pathway
  is the sign of its genes.

This **costs** validation loss relative to the shipped dense term — +2.8 nats on
`Zavidij_2020`, +9.7 on `Foster_2026`. A dense per-gene matrix describes one
large diffuse shift more cheaply than 1,615 annotated units can. The
justification is not fit; it is that the real annotation then beats a
size-matched shuffle by +5.9 and +10.7 nats, against +0.6 and +1.3 before.

In [ ]:
ARCH, L2_LAMBDA = mm.ARCH, mm.L2_LAMBDA
latent_dim = mm.latent_dim_for(len(pathway_names))
adj_tensor = torch.tensor(adj_df.values, dtype=torch.float32)

set_all_seeds(SEED)
model = InformedVAE(adj=adj_tensor, latent_dim=latent_dim, seed=SEED,
                    l2_lambda=L2_LAMBDA, likelihood="nb", n_cov=n_cov, **ARCH)
print("architecture:", ARCH)
print(f"units              {len(pathway_names):,}    latent dim {latent_dim:,}")
print(f"covariate term     cov_path {tuple(model.decoder.cov_path.weight.shape)} "
      f"(units x cov), gene-level term present: "
      f"{hasattr(model.decoder, 'cov_decoder')}")

EPOCHS = int(epochs)
CKPT = OUT_DIR / "model.pt"
FINGERPRINT = OUT_DIR / "model_fingerprint.json"
# The architecture is part of the fingerprint. Without it a checkpoint from the
# shipped decoder would be adopted by this notebook and read as though it came
# from this one -- which is exactly the confusion the whole rebuild exists to
# remove.
design = {"study": STUDY, "epochs": EPOCHS, "seed": SEED,
          "blocks": list(blocks), "n_cov": int(n_cov),
          "analysed_only": bool(analysed_only),
          "architecture": {k: str(v) for k, v in sorted(ARCH.items())},
          "cells": int(adata.n_obs), "train_cells": int(len(idx_tr))}
reusable = (not force_retrain and CKPT.exists() and FINGERPRINT.exists()
            and json.loads(FINGERPRINT.read_text()) == design)

if reusable:
    model.load_state_dict(torch.load(CKPT, map_location="cpu"))
    model = model.to(DEVICE)
    hist = pd.read_csv(OUT_DIR / "training_history.csv")
    # None, not 0.0: this fit was adopted, so this run has no training time to
    # report and a zero would read as "trained instantly".
    train_min = None
    DEV = next(model.parameters()).device
    print(f"\nreused the saved fit in {OUT_DIR} -- the design fingerprint "
          "matches exactly.\nSet force_retrain=True to train anyway.")
else:
    t0 = time.time()
    model, history = train_ivae_modern(
        model, x_train=data["x_train"], x_val=data["x_val"],
        x_counts_train=data["counts_train"], x_counts_val=data["counts_val"],
        cov_train=data["cov_train"], cov_val=data["cov_val"], device=DEVICE,
        epochs=EPOCHS, patience=EPOCHS, batch_size=BATCH_SIZE, lr=1e-3,
        weight_decay=1e-6, warmup_epochs=10, max_grad_norm=1.0)
    train_min = (time.time() - t0) / 60
    DEV = next(model.parameters()).device
    hist = pd.DataFrame(history)
    print(f"\ntrained {len(hist)} epochs in {train_min:.1f} min, best val "
          f"{hist['val'].min():.2f} at epoch {int(hist['val'].idxmin()) + 1}")
    torch.save(model.state_dict(), CKPT)
    hist.to_csv(OUT_DIR / "training_history.csv", index=False)
    FINGERPRINT.write_text(json.dumps(design, indent=2, sort_keys=True))
best_val = float(hist["val"].min())

## 4. Four checks, all pass/fail

Nothing in section 5 is readable unless all four hold, so they are `assert`s
rather than numbers to be eyeballed.

1. **No inverted encoder unit.** `nonneg_encoder` reparameterises every encoder
   weight as `softplus(theta) * mask`, so a negative live weight is
   unrepresentable. Must be exactly zero.
2. **No negative decoder weight.** The same for `dec_out`. On the shipped fit
   this was 0.502 in `Zavidij_2020` and 0.503 in `Foster_2026`, with a per-unit
   positive fraction whose median was exactly 0.500 — which is why there was no
   pathway-level sign to read and why a post-hoc flip could not recover one.
3. **No unit writes against itself.** Every unit's own direction matches the
   direction it writes to its member genes. Monotone `tanh` and positive
   `w_sum` make a flip impossible, so this is a check that the fit is what it
   claims. A unit can still write *nothing*: `tanh` saturates, and those dead
   units are counted rather than hidden.
4. **`pathway_contrast` equals the forward pass.** The read-out reads a weight
   matrix; the model computes a forward pass. They must agree to float32
   precision or the read-out is describing a different model.

In [ ]:
# Pre-activations for every cell under its own covariate, in float64. The cast
# matters: `tanh(pre + d) - tanh(pre)` for a small `d` is a catastrophic
# cancellation and underflows to exactly 0 in float32 for any unit with |pre|
# past about 4, which silently turns a unit's direction into no direction.
# Computed once here, because every contrast below is an analytic function of it:
# the covariate does not reach the encoder, so `z` -- and therefore
# `dec_latent(z)` -- is the same under every counterfactual stage.
model.eval()
chunks, sat_chunks = [], []
with torch.no_grad():
    for lo in range(0, len(x_trans), 8192):
        hi = min(lo + 8192, len(x_trans))
        xb = torch.tensor(x_trans.iloc[lo:hi].values, dtype=torch.float32, device=DEV)
        cb = torch.tensor(cov.iloc[lo:hi].values, dtype=torch.float32, device=DEV)
        z = model.encode(xb, cb)[0]
        pre = (model.decoder.dec_latent(z) + model.decoder.cov_path(cb)).double()
        chunks.append(pre.cpu().numpy())
        sat_chunks.append((torch.tanh(pre).abs() > 0.99).double().cpu().numpy())
PRE = np.concatenate(chunks)
saturated = np.concatenate(sat_chunks).mean(axis=0)
del chunks, sat_chunks
print(f"pre-activations    {PRE.shape[0]:,} cells x {PRE.shape[1]:,} units, "
      f"float64 ({PRE.nbytes / 1e9:.2f} GB)")

W_path = model.decoder.cov_path.weight.detach().cpu().numpy()
w_out = model.decoder.dec_out.effective_weight().detach().double().cpu().numpy()
members = model.decoder.dec_out.mask.detach().double().cpu().numpy()
n_members = members.sum(0)
w_sum = (w_out * members).sum(0)

sign = unit_sign_structure(model, min_genes=10)
assert sign["encoder_frac_negative"] == 0.0, (
    f"{sign['encoder_frac_negative']:.4f} of encoder live weights are negative; "
    "nonneg_encoder is not doing what it claims")
assert sign["dec_out_frac_negative"] == 0.0, (
    f"{sign['dec_out_frac_negative']:.4f} of dec_out live weights are negative; "
    "nonneg_decoder is not doing what it claims and no unit sign below means "
    "anything")
assert (w_sum >= 0).all(), "softplus produced a negative weight sum"

print(f"encoder live weights negative   {sign['encoder_frac_negative']:.4f}"
      "   (must be 0)")
print(f"dec_out live weights negative   {sign['dec_out_frac_negative']:.4f}"
      "   (must be 0; the shipped fit was 0.50)")
sat_half = int((saturated > 0.5).sum())
print(f"saturation, max over units      {saturated.max():.4f}")
print(f"units saturated in >half cells  {sat_half:,} of {len(pathway_names):,}"
      f" ({sat_half / len(pathway_names):.0%})"
      "   <- see section 5 on what this costs")

## 5. The stage read-out: one signed number per unit

For one lineage, the effect of being at stage *b* rather than *a* — holding
phenotype fixed — is

```
delta_pre = cov_path[:, stage_<lineage>__<b>] - cov_path[:, stage_<lineage>__<a>]
```

a shift on each unit's pre-activation. **No cell is re-encoded and no cell is
re-weighted**, so a change in which phenotypes were sampled at a stage cannot
produce a contrast. That is the whole reason for conditioning, and it is the
same argument the previous notebook made — the difference is that this vector
has 1,615 entries indexed by pathway, not 2,000-odd indexed by gene.

Two columns per contrast, and the difference matters:

- **`delta_pre`** is that exact weight difference, cell-independent.
- **`member_shift_z`** is what reaches the genes: `delta_post * w_sum /
  sqrt(n_genes)`, where `delta_post` is the mean change in `tanh(pre)` over the
  *healthy* cells of that lineage — what would happen to these cells if they
  were at the later stage. The `1/sqrt(n)` corrects a size tilt that otherwise
  hands the top of the ranking to the smallest units (Spearman between
  |statistic| and set size falls from −0.49 to −0.09).

The ranking is by `member_shift_z`, the same statistic `01_architecture.py`
scored every arm on.

### What saturation costs, and why section 4 counts it

`tanh` saturates, and this architecture saturates far harder than the shipped
one. Measured on the 250-epoch `Zavidij_2020` fits: **80% of units sit in the
flat tail for more than half their cells, against 24% under the shipped
decoder**, and the median |`delta_post`| falls from 5.0e-02 to 2.2e-03 — a
factor of 23. A pinned unit writes nothing to its genes whatever its
`delta_pre` is, so `member_shift_z` is in effect a ranking over the units that
still have headroom.

That is why it beats ranking on the weights alone — in `01_architecture.py`
the weight ranking put the target at 56 and 19 while `member_shift_z` put it
at 2 and 1 — and it is equally the honest caveat on it: **a unit can be
biologically involved and invisible here because it is pinned.** The count
travels with the numbers in `summary.json`, and `FINDINGS.md` states it beside
the likelihood cost rather than under it.

### Two parameterisations, and no count of either

Each lineage separately, and a cell-count-weighted **pooled** contrast — what a
plain, non-interacted `cohort` block would have estimated. Reporting both
answers "is this shared across lineages, or is it one lineage showing through
an average".

Reactome C2:CP **nests strictly** — `INTERFERON_ALPHA_BETA_SIGNALING` inside
`INTERFERON_SIGNALING` inside `CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM` — so "how
many units responded" counts one family many times, and an overlap p-value over
units is not a test of anything. This notebook reports **signed per-unit
contrasts and ranks only**. The one place a p-value appears in this experiment
is `03_donor_test.py`, where the sampling unit is the donor.

In [ ]:
cell_counts = (adata.obs.groupby(["subset", "cohort"], observed=True)
               .size().to_dict())
present_lin = [s for s in mm.SUBSETS
               if any((s, st) in cell_counts for st in STAGES)]
LINEAGES = ["pooled"] + [s for s in mm.MAJOR_SUBSETS if s in present_lin]
lineage_of = adata.obs["subset"].astype(str).to_numpy()
stage_of = adata.obs["cohort"].astype(str).to_numpy()


def baseline_rows(lineage: str) -> np.ndarray:
    """The healthy cells the post-activation shift is evaluated on."""
    m = stage_of == STAGES[0]
    if lineage != "pooled":
        m &= lineage_of == lineage
    return np.where(m)[0]


def contrast(lineage: str, frm: str, to: str):
    """`delta_pre` from the weights, `delta_post` from those cells. None if absent."""
    if lineage == "pooled":
        d_pre, weights = mm.pooled_stage_contrast(W_path, cov_columns,
                                                  cell_counts, frm, to)
        if d_pre is None:
            return None
    else:
        d_pre = mm.stage_contrast_weights(W_path, cov_columns, lineage, frm, to)
        if d_pre is None:
            return None
        # The public read-out, exercised rather than reimplemented: build the two
        # covariate settings and let the model difference them. Asserting the two
        # agree is what makes `stage_contrast_weights` safe to use for `pooled`,
        # where no single pair of covariate columns describes the contrast.
        a = np.zeros(n_cov, dtype=np.float32)
        b = np.zeros(n_cov, dtype=np.float32)
        a[cov_columns.index(mm.stage_col(lineage, frm))] = 1.0
        b[cov_columns.index(mm.stage_col(lineage, to))] = 1.0
        api = model.pathway_contrast(a, b).detach().cpu().numpy()
        dev = float(np.abs(api - d_pre).max())
        assert dev < 1e-4, (
            f"pathway_contrast and the weight difference disagree by {dev:.2e} "
            f"for {lineage} {frm}->{to}")
    rows = baseline_rows(lineage)
    if len(rows) < 8:
        return None
    base = PRE[rows]
    d_post = (np.tanh(base + d_pre) - np.tanh(base)).mean(axis=0)
    return d_pre, d_post, len(rows)


records, api_devs = [], []
for lineage in LINEAGES:
    for frm, to in LADDER:
        got = contrast(lineage, frm, to)
        if got is None:
            continue
        d_pre, d_post, n_base = got
        records.append(pd.DataFrame({
            "unit": pathway_names, "lineage": lineage,
            "contrast": f"{frm}->{to}", "n_genes": n_members.astype(int),
            "delta_pre": d_pre, "delta_post": d_post,
            "member_shift_z": d_post * w_sum / np.sqrt(np.maximum(n_members, 1)),
            "baseline_cells": n_base}))
contrasts = pd.concat(records, ignore_index=True)

disagree = int((np.sign(contrasts["delta_pre"])
                * np.sign(contrasts["member_shift_z"]) < 0).sum())
dead = int((contrasts["member_shift_z"] == 0).sum())
assert disagree == 0, (
    f"{disagree} unit-contrasts write to their genes in the direction opposite "
    "to their own shift; under nonneg_decoder that is impossible")

contrasts["rank"] = (contrasts.groupby(["lineage", "contrast"])["member_shift_z"]
                     .rank(ascending=False, method="first").astype(int))
contrasts.to_csv(OUT_DIR / "unit_contrasts.csv", index=False)

# Wide view over the endpoint, which is what every figure orders rows by.
END = LADDER_NAMES[-1]
wide_end = (contrasts[contrasts["contrast"] == END]
            .pivot(index="unit", columns="lineage", values="member_shift_z"))
MAJOR = [l for l in mm.MAJOR_SUBSETS if l in wide_end.columns]
ORDER = list(wide_end[MAJOR].mean(axis=1).sort_values(ascending=False).index)

print(f"{len(contrasts):,} unit-contrasts: {len(pathway_names):,} units x "
      f"{contrasts['lineage'].nunique()} lineages x {len(LADDER_NAMES)} "
      "contrasts")
print(f"units writing against themselves      {disagree}   (must be 0)")
print(f"unit-contrasts reaching nothing       {dead:,} of {len(contrasts):,} "
      "(saturated units)")
print(f"median |delta_post| over contrasts    "
      f"{contrasts['delta_post'].abs().median():.2e}   (5.0e-02 on the shipped "
      "decoder)")
print(f"\nendpoint {END}, top 12 by the CD4/CD8 mean:\n")
print(wide_end.loc[ORDER[:12]].rename(index=mm.short).round(4).to_string())
print(f"\nand the 6 most strongly down:\n")
print(wide_end.loc[ORDER[-6:]].rename(index=mm.short).round(4).to_string())

### What the shuffled annotation gives, for the same model

`01_architecture.py` fits this architecture on a **row-permuted** adjacency:
every unit keeps its size, its name and the entire pathway-by-pathway overlap
matrix, and only which named genes sit in a unit changes. Its numbers are read
here rather than recomputed, so the two scripts cannot disagree.

In [ ]:
arch_arms = {}
if ARCH_SUMMARY.exists():
    arms = json.loads(ARCH_SUMMARY.read_text()).get("arms", {})
    for arm in ("real_shipped", "perm_shipped", "real_signed", "perm_signed"):
        if arm not in arms:
            continue
        a = arms[arm]
        arch_arms[arm] = {
            "best_val": a.get("best_val"),
            "dec_out_frac_negative": a.get("dec_out_frac_negative"),
            # Per lineage, not per arm: the stage block is interacted with
            # lineage, so one arm yields one rank per lineage.
            "unit_rank_target": {l: v.get("unit_rank_target")
                                 for l, v in a.get("lineages", {}).items()},
            "gene_rank_interferon_best": {
                l: v.get("gene_rank_interferon_best")
                for l, v in a.get("lineages", {}).items()},
        }
    hdr = (f"{'arm':<14}{'val':>10}{'unit rank':>16}{'gene rank':>14}"
           f"{'dec_out -ve':>13}")
    print(hdr)
    for arm, r in arch_arms.items():
        units_s = ", ".join(str(v) for v in r["unit_rank_target"].values())
        genes_s = ", ".join(str(v) for v in
                            r["gene_rank_interferon_best"].values())
        print(f"{arm:<14}{r['best_val']:>10.1f}{units_s:>16}{genes_s:>14}"
              f"{r['dec_out_frac_negative']:>13.3f}")
    print("\nthe gene-level column is the previous notebook's read-out. It ranks "
          "the same\nprogramme first whether the annotation is real or shuffled, "
          "which is why it is\nevidence about the read-out rather than a result.")
else:
    print(f"{ARCH_SUMMARY} not found -- run `snakemake architecture` first.")

## 6. Do the two lineages agree? A power diagnostic

CD4 and CD8 are separate columns of the same design, so they are two
near-independent estimates of the same disease effect. How well they agree is
therefore a **diagnostic on this fit** rather than a biological claim: if
agreement tracks the cell count of the thinnest arm, what you are looking at is
noise attenuation, not lineages decoupling. Computed over all units, because a
correlation on a short list has a uselessly wide interval.

In [ ]:
from scipy import stats

CD4, CD8 = "CD4", "CD8"
cells_by = (adata.obs.groupby(["subset", "cohort"], observed=True).size()
            .unstack().reindex(columns=STAGES).fillna(0).astype(int))
donors_by = (adata.obs.groupby(["subset", "cohort"], observed=True)["donor_id"]
             .nunique().unstack().reindex(columns=STAGES).fillna(0).astype(int))
rng = np.random.default_rng(SEED)
pivot = {c: (contrasts[contrasts["contrast"] == c]
             .pivot(index="unit", columns="lineage", values="member_shift_z"))
         for c in LADDER_NAMES}

agree_rows = []
for label in LADDER_NAMES:
    w = pivot[label]
    if CD4 not in w.columns or CD8 not in w.columns:
        continue
    x, y = w[CD4].to_numpy(), w[CD8].to_numpy()
    rho = stats.spearmanr(x, y).statistic
    bs = [stats.spearmanr(x[i], y[i]).statistic
          for i in (rng.integers(0, len(x), len(x)) for _ in range(1000))]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    frm, to = label.split("->")
    lim_c = int(min(cells_by.loc[l, s] for l in (CD4, CD8) for s in (frm, to)
                    if s in cells_by.columns))
    lim_d = int(min(donors_by.loc[l, s] for l in (CD4, CD8) for s in (frm, to)
                    if s in donors_by.columns))
    agree_rows.append({"contrast": label, "rho": round(float(rho), 3),
                       "ci_lo": round(float(lo), 3), "ci_hi": round(float(hi), 3),
                       "limiting cells": lim_c, "limiting donors": lim_d})
agreement = pd.DataFrame(agree_rows).set_index("contrast")
agreement.to_csv(OUT_DIR / "table3_lineage_agreement.csv")
print("Table 3. CD4 against CD8 over all units, with a 1,000-draw bootstrap "
      "interval\n")
print(agreement.to_string())

## 7. Inside the leading units: which genes move them

A within-unit question, and the only place gradients appear. Two numbers that
sound alike and measure different things:

- **Encoder side, integrated gradients.** Which genes move the *activation* —
  the unit's sensitivity, what it reads the pathway through.
- **Decoder side, member share.** `w_out[gene, unit] / sum over members`, the
  share of the unit's written response each member carries. Under
  `nonneg_decoder` this is a genuine share: every term is positive and they sum
  to one, which was not true of the shipped decoder.

They routinely disagree and name different genes. A unit can read a broad
response through a handful of members. Report both; never quote one as the
other. The dashed line is the share a member would carry if all contributed
equally.

In [ ]:
FOCUS = ORDER[:4]
late = stage_of == STAGES[-1]
driver_rows, driver_detail = [], {}
for unit in FOCUS:
    j = pathway_names.index(unit)
    sel = np.where(late)[0][:256]
    if len(sel) < 8:
        sel = np.arange(min(256, len(x_trans)))
    attr = integrated_gradients(
        model,
        x=torch.tensor(x_trans.iloc[sel].values, dtype=torch.float32, device=DEV),
        pathway_idx=j,
        cov=torch.tensor(cov.iloc[sel].values, dtype=torch.float32, device=DEV),
        steps=50)
    ga = np.abs(attr.detach().cpu().numpy()).mean(axis=0)
    # Only genes the unit actually reads. The mask puts every live encoder
    # weight on a member, so most of the panel attributes exactly zero and
    # `nlargest` would fill the figure with alphabetical ties at 0.
    enc = pd.Series(ga / max(ga.sum(), 1e-30), index=gene_names)
    enc = enc[enc > 0].nlargest(8)
    live = members[:, j] > 0
    dec = pd.Series(w_out[live, j] / max(w_out[live, j].sum(), 1e-30),
                    index=[g for g, m in zip(gene_names, live) if m]).nlargest(8)
    driver_detail[unit] = {"encoder": enc, "decoder": dec}
    driver_rows.append({
        "unit": unit, "n_genes": int(n_members[j]),
        "member_shift_z_endpoint": float(wide_end.loc[unit, MAJOR].mean()),
        "encoder_top": enc.index[0], "encoder_top_share": round(float(enc.iloc[0]), 4),
        "decoder_top": dec.index[0], "decoder_top_share": round(float(dec.iloc[0]), 4),
        "sides_agree": enc.index[0] == dec.index[0],
        "even_share": round(1.0 / max(int(n_members[j]), 1), 4)})
drivers = pd.DataFrame(driver_rows).set_index("unit")
drivers.to_csv(OUT_DIR / "table2_drivers.csv")
n_agree = int(drivers["sides_agree"].sum())
print(f"Table 2. The four leading units, from both sides "
      f"({n_agree} of {len(drivers)} agree on the top gene)\n")
print(drivers.rename(index=mm.short).to_string())

## 8. The reference frame, which is a picture

Standard scanpy from raw counts — `normalize_total`, `log1p`, 2,000 HVGs, scale,
50 PCs, neighbours, UMAP. It is here so the activations can be looked at, and
**no claim in this notebook rests on it.** Cells in these frames group
substantially by donor, and cohort is a donor property, so any apparent
separation by stage in a picture like this is donor-shaped structure, not
evidence about stage.

In [ ]:
# The *pre*-activation, not `tanh` of it. Under this architecture the units sit
# deep in the flat tail -- `tanh(pre)` is 1.000 for most cells -- so a figure
# coloured by the activation is one flat colour and hides the structure that is
# there. The pre-activation is unbounded and shows it.
LEAD = ORDER[0]
lead_pre = PRE[:, pathway_names.index(LEAD)].copy()
del PRE          # 0.8 GB on the larger study, and nothing below needs it
all_obs = adata.obs[["cohort", "subset", "pheno", "donor_id"]].copy()
ref = mm.standard_umap(counts_df, all_obs, seed=SEED)
U = np.asarray(ref.obsm["X_umap"], dtype=np.float32)
print(f"frame              {U.shape[0]:,} cells, leading unit {mm.short(LEAD)}")

## 9. Figures

Four conventions, applied throughout:

- **Signed quantities get a diverging scale with a neutral midpoint.** Zero
  means "no change", so zero is near-white and the two directions are two hues.
  A sequential ramp on signed data hides the sign.
- **Identity is never colour alone.** The lineage panel is directly labelled at
  the cluster centroids.
- **Three categorical hues at most in any one scatter.** Five lineages cannot be
  told apart reliably — the magenta/orange pair sits at normal-vision ΔE 12.9,
  below the readable floor — so the small lineages, which are held out of every
  read-out anyway, fold into one "other" class.
- **No significance mark anywhere in this notebook.** There is no test over
  units, by design; the donor-level test is `03_donor_test.py`.

In [ ]:
from matplotlib.colors import TwoSlopeNorm

C_BLUE, C_ORANGE, C_AQUA = mm.C_BLUE, mm.C_ORANGE, mm.C_AQUA
C_GREY, C_FAINT = mm.C_GREY, mm.C_FAINT
DIVERGING = mm.diverging()

lineage = all_obs["subset"].astype(str)
MAJOR_PRESENT = [s for s in mm.MAJOR_SUBSETS if s in set(lineage)]
lin3 = lineage.where(lineage.isin(MAJOR_PRESENT), "other")
LIN_COLOURS = {k: mm.LINEAGE_COLOURS[k] for k in MAJOR_PRESENT} | {"other": C_AQUA}


def scatter_cat(ax, labels, colours, *, title="", size=2):
    for name, colour in colours.items():
        m = (labels == name).to_numpy()
        if not m.any():
            continue
        ax.scatter(U[m, 0], U[m, 1], s=size, c=colour, linewidths=0, alpha=0.75)
        ax.annotate(f"{name}  n={int(m.sum()):,}",
                    (np.median(U[m, 0]), np.median(U[m, 1])), fontsize=8,
                    fontweight="bold", ha="center", color="#0b0b0b",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none",
                              alpha=0.75))
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)


def scatter_cont(ax, values, *, title="", cmap="Blues", centre=False, label=""):
    norm = None
    if centre:
        span = float(np.nanquantile(np.abs(values), 0.99)) or 1e-9
        norm = TwoSlopeNorm(vmin=-span, vcenter=0.0, vmax=span)
    sct = ax.scatter(U[:, 0], U[:, 1], s=2, c=values, cmap=cmap, norm=norm,
                     linewidths=0)
    plt.colorbar(sct, ax=ax, shrink=0.8, label=label or title)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

### Figure 1. The reference frame

Lineage is directly labelled rather than relying on colour. The two continuous
panels are the atlas's own naive and cytotoxicity scores, which show that the
frame carries the differentiation axis — the thing the model conditions on —
without needing nineteen colours for nineteen phenotypes.

In [ ]:
have = [c for c in ["Cytotoxicity", "Naive"] if c in adata.obs.columns]
fig, axes = plt.subplots(1, 1 + len(have), figsize=(5.2 * (1 + len(have)), 4.6))
axes = np.atleast_1d(axes)
scatter_cat(axes[0], lin3, LIN_COLOURS, title="T-cell lineage")
for ax, col in zip(axes[1:], have):
    scatter_cont(ax, adata.obs[col].to_numpy(), title=col.lower(), cmap="Blues")
fig.suptitle(f"{STUDY}: reference frame, standard scanpy from raw counts, "
             f"{len(U):,} cells", y=1.02)
fig.tight_layout(); fig.savefig(FIGS / "fig1_frame.png", dpi=300); plt.show()

### Figure 2. The ladder, over the model's own units

Every column is against the same `Non` baseline, so a row reads left to right
as a trajectory. Rows are the 25 units with the highest and the 10 with the
lowest endpoint mean over CD4 and CD8 — a deterministic order that uses no
test, because there is no test over units to use.

**Reactome nests**, so neighbouring rows are often parent and child of one
family rather than independent findings. Read the families, not the row count.

In [ ]:
PANEL = ORDER[:25] + ORDER[-10:]
cols = [(lin, c) for lin in MAJOR for c in LADDER_NAMES]
M = pd.DataFrame({f"{lin}|{c}": pivot[c][lin].reindex(PANEL)
                  for lin, c in cols if lin in pivot[c].columns})
span = float(np.abs(M.to_numpy()).max()) or 1e-9

fig, ax = plt.subplots(figsize=(1.0 * M.shape[1] + 4.6, 0.27 * len(PANEL) + 1.9))
ax.imshow(M.to_numpy(), aspect="auto", cmap=DIVERGING,
          norm=TwoSlopeNorm(vmin=-span, vcenter=0.0, vmax=span))
for k in range(1, len(MAJOR)):
    ax.axvline(k * len(LADDER_NAMES) - 0.5, color="white", lw=3)
ax.axhline(24.5, color="#0b0b0b", lw=1.1, ls=":")
ax.set_xticks(range(M.shape[1]))
ax.set_xticklabels([c.split("|")[1].replace("->", "→") for c in M.columns],
                   fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(PANEL)))
ax.set_yticklabels([mm.short(p)[:56] for p in PANEL], fontsize=7)
for k, lin in enumerate(MAJOR):
    ax.annotate(lin, ((k + 0.5) * len(LADDER_NAMES) - 0.5, -1.4), fontsize=10,
                fontweight="bold", ha="center", annotation_clip=False)
ax.tick_params(length=0); ax.grid(False)
plt.colorbar(ax.images[0], ax=ax, shrink=0.55, label="member_shift_z")
fig.suptitle(f"{STUDY}: the units the stage sequence moves "
             f"(top 25 and bottom 10 of {len(pathway_names):,})", y=1.01)
fig.savefig(FIGS / "fig2_ladder.png", dpi=300, bbox_inches="tight"); plt.show()

### Figure 3. CD4 against CD8

One point per unit. If the lineages moved together, points would lie on the
diagonal. The annotation carries the sample size that limits each panel,
because a weak correlation from a thin arm and a weak correlation from genuine
independence look identical here.

In [ ]:
panels = [c for c in LADDER_NAMES if c in agreement.index]
fig, axes = plt.subplots(1, len(panels), figsize=(3.5 * len(panels), 3.7),
                         sharex=True, sharey=True)
for ax, label in zip(np.atleast_1d(axes), panels):
    w = pivot[label]
    x, y = w[CD4].to_numpy(), w[CD8].to_numpy()
    ax.scatter(x, y, s=5, c=C_FAINT, linewidths=0)
    ax.axhline(0, color=C_GREY, lw=0.7); ax.axvline(0, color=C_GREY, lw=0.7)
    lo, hi = np.nanpercentile(np.r_[x, y], [0.5, 99.5])
    ax.plot([lo, hi], [lo, hi], color=C_ORANGE, lw=1.2, ls="--")
    row = agreement.loc[label]
    ax.set_title(f"{label.replace('->', chr(8594))}\nrho {row['rho']:+.2f}   "
                 f"limiting n = {int(row['limiting cells']):,} cells", fontsize=9)
    ax.set_xlabel("CD4 member_shift_z", fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("CD8 member_shift_z", fontsize=8)
fig.suptitle(f"{STUDY}: lineage agreement against the healthy baseline "
             "(dashed = equality)", y=1.03)
fig.tight_layout()
fig.savefig(FIGS / "fig3_cd4_vs_cd8.png", dpi=300, bbox_inches="tight"); plt.show()

### Figure 4. Which genes drive each unit, from both sides

**Left (encoder):** integrated gradients — which genes move the unit's
activation. **Right (decoder):** each member's share of the unit's written
response. Bars far beyond the dashed equal-share line mean the unit rests on a
few genes.

In [ ]:
import textwrap

fig, axes = plt.subplots(len(FOCUS), 2, figsize=(9.2, 2.3 * len(FOCUS)))
axes = np.atleast_2d(axes)
for i, unit in enumerate(FOCUS):
    detail = driver_detail[unit]
    even = 1.0 / max(int(n_members[pathway_names.index(unit)]), 1)
    for j, (side, colour) in enumerate([("encoder", C_BLUE), ("decoder", C_ORANGE)]):
        ser = detail[side].sort_values()
        ax = axes[i, j]
        ax.barh(range(len(ser)), ser.to_numpy(), color=colour, height=0.68)
        ax.set_yticks(range(len(ser)))
        ax.set_yticklabels(ser.index, fontsize=7)
        ax.tick_params(length=0, labelsize=7)
        ax.grid(axis="y", visible=False)
        ax.axvline(even, color=C_GREY, ls="--", lw=1)
        if i == 0:
            ax.set_title("encoder: what moves the unit" if side == "encoder"
                         else "decoder: what the unit writes", fontsize=9)
        if j == 0:
            ax.set_ylabel("\n".join(textwrap.wrap(mm.short(unit), 22)),
                          fontsize=7.5)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
fig.suptitle(f"{STUDY}: the four leading units, from both sides "
             "(dashed = equal share)", y=1.005)
fig.tight_layout()
fig.savefig(FIGS / "fig4_drivers.png", dpi=300, bbox_inches="tight"); plt.show()

### Figure 5. The leading unit on the frame, one panel per stage

The coordinates are identical in every panel — only which cells are drawn in
colour changes. Colour is the leading unit's activation on a diverging scale
centred at zero.

**Read this as a picture of where the signal sits, not as evidence that the
stages differ.** Every donor belongs to exactly one stage, so stage-shaped
structure here is donor-shaped structure. The quantitative statement is in
section 5, where the contrast is taken at fixed phenotype.

The colour is the *pre*-activation, centred on its own median. `tanh` of it is
1.000 for most of these cells, so the activation itself is one flat colour.

In [ ]:
# Centred on the unit's own median rather than on zero: it sits far from zero
# in every cell, so a scale centred at zero paints every panel one colour.
centre = float(np.median(lead_pre))
span = float(np.nanquantile(np.abs(lead_pre - centre), 0.99)) or 1e-9
fig, axes = plt.subplots(1, len(STAGES), figsize=(4.2 * len(STAGES), 4.2),
                         sharex=True, sharey=True)
for ax, stage in zip(np.atleast_1d(axes), STAGES):
    m = stage_of == stage
    ax.scatter(U[~m, 0], U[~m, 1], s=1.5, c=C_FAINT, linewidths=0)
    sct = ax.scatter(U[m, 0], U[m, 1], s=2.5, c=lead_pre[m], cmap=DIVERGING,
                     norm=TwoSlopeNorm(vmin=centre - span, vcenter=centre,
                                       vmax=centre + span),
                     linewidths=0)
    n_don = int(all_obs.loc[m, "donor_id"].nunique())
    ax.set_title(f"{stage}\n{int(m.sum()):,} cells, {n_don} donors", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(sct, ax=np.atleast_1d(axes)[-1], shrink=0.8,
             label="unit pre-activation")
fig.suptitle(f"{STUDY}: {mm.short(LEAD)} pre-activation, by stage", y=1.02)
fig.tight_layout()
fig.savefig(FIGS / "fig5_lead_by_stage.png", dpi=300, bbox_inches="tight")
plt.show()

### Table 1. The design, and the power behind it

Cells and donors per stage, split by lineage. Every number above is limited by
the thinnest arm in this table.

In [ ]:
by_stage = adata.obs.groupby("cohort", observed=True)
per_lin = (adata.obs.groupby(["cohort", "subset"], observed=True).size()
           .unstack(fill_value=0).reindex(STAGES, fill_value=0))
t1 = pd.DataFrame({"donors": by_stage["donor_id"].nunique(),
                   "cells": by_stage.size(),
                   "median genes/cell": by_stage["n_genes"].median()}).reindex(STAGES)
for lin in mm.MAJOR_SUBSETS:
    if lin in per_lin.columns:
        t1[f"{lin} cells"] = per_lin[lin]
t1 = t1.fillna(0).astype(int)
t1.index.name = "stage"
t1.to_csv(OUT_DIR / "table1_design.csv")
thin = int(t1[[f"{l} cells" for l in mm.MAJOR_SUBSETS
               if f"{l} cells" in t1.columns]].min().min())
print("Table 1. Design\n")
print(t1.to_string())
dropped = [b for b in CANDIDATE_BLOCKS if b not in accepted]
print(f"\n  covariates entered: {blocks}")
print(f"  covariates dropped: {dropped}  (reasons in covariate_audit.csv)")
print(f"  thinnest lineage x stage arm: {thin:,} cells -- the ceiling on every "
      "contrast that\n    involves that stage")

## 10. What this run says, before looking at any paper

`summary.json` is the machine-readable version of everything above. It is what
`04_figures.py`, `FINDINGS.md` and `verify_findings.py` read. The literature
check belongs there, after **both** studies have been run.

### How to read this run, in order

Be willing to stop at any of these:

1. **`covariate_blocks_dropped` — is it empty?** If a block was dropped here
   and not in the other study, the two fits are not comparable and nothing
   cross-study can be said. That is what the whole of section 2 exists to make
   visible.
2. **`encoder_frac_negative` and `dec_out_frac_negative` — both zero?** If not,
   some unit represents its pathway upside-down, or writes its response in both
   directions at once, and every sign above is meaningless.
3. **`units_sign_disagree` — zero?** It is impossible under this architecture,
   so a non-zero value means the fit is not what it claims. Beside it,
   **`units_saturated_gt_half`**: this architecture pushes most units into the
   flat tail of `tanh`, where they write nothing, so the ranking in section 5
   is a ranking over the units that still have headroom. That is a limit on
   what can be *found*, not a reason to distrust what was.
4. **`perm_unit_rank_target` against `real_unit_rank_target`**, from
   `01_architecture.py`. The gap is what the biology in the map is worth. A
   model that scores the same on a shuffled map is not using its annotation,
   whatever its ranking looks like.
5. **`top_units`** — the result, as a signed ranking. Two things bound it. The
   sampling unit is the **cell**, not the donor: these contrasts are estimated
   from between-donor variation with no within-donor stage replication, so a
   programme leading this list on three donors leads it on three donors.
   `03_donor_test.py` is where that is tested, and on the previous read-out it
   overturned the cross-study replication entirely. And **Reactome nests**, so
   the top of this list is families, not independent hits.

In [ ]:
summary = {
    "study": STUDY, "cells": int(adata.n_obs),
    "donors": int(adata.obs["donor_id"].nunique()),
    "stages": STAGES, "ladder_contrasts": LADDER_NAMES,
    "lineages": LINEAGES,
    "covariate_blocks_entered": blocks,
    "covariate_blocks_dropped": dropped,
    "covariate_blocks_audited": CANDIDATE_BLOCKS,
    "n_cov": int(n_cov), "epochs": int(EPOCHS), "seed": SEED,
    "architecture": {k: str(v) for k, v in sorted(ARCH.items())},
    "mask_units": int(len(pathway_names)),
    "fit_reused": train_min is None,
    "train_minutes": None if train_min is None else round(train_min, 2),
    "best_val_loss": round(best_val, 3),
    "encoder_frac_negative": sign["encoder_frac_negative"],
    "dec_out_frac_negative": sign["dec_out_frac_negative"],
    "units_sign_disagree": disagree,
    "unit_contrasts_reaching_nothing": dead,
    "saturation_max": round(float(saturated.max()), 4),
    "units_saturated_gt_half": sat_half,
    "delta_post_abs_median": round(
        float(contrasts["delta_post"].abs().median()), 6),
    "thinnest_arm_cells": thin,
    "endpoint_contrast": END,
    "lead_unit": LEAD,
    "top_units": [mm.short(p) for p in ORDER[:10]],
    "top_unit_z": {mm.short(p): round(float(wide_end.loc[p, MAJOR].mean()), 4)
                   for p in ORDER[:10]},
    "bottom_units": [mm.short(p) for p in ORDER[-5:]],
    "lineage_agreement": {k: float(v) for k, v in agreement["rho"].items()},
    "driver_units": [mm.short(p) for p in FOCUS],
    "driver_sides_agree_on_top_gene": f"{n_agree} of {len(drivers)}",
    "architecture_arms": arch_arms,
    # Per lineage, from act 1. The target is the unit the retired gene-level
    # baseline points at, not a known answer -- myeloma has none. It is here so
    # the two architectures can be compared on a fixed unit.
    "real_unit_rank_target": arch_arms.get("real_signed", {}).get(
        "unit_rank_target", {}),
    "perm_unit_rank_target": arch_arms.get("perm_signed", {}).get(
        "unit_rank_target", {}),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))